# Stage 9a ablations — single-fold matrix (Kaggle T4)

Three reviewer-grade ablations run on **fold 0 only** (Stage 9a's hardest fold at 0.5681, best signal-to-noise for ablations) over the locked Stage 1 v2 / Stage 9a recipe.  Every other knob is fixed.

| Variant | Changes | Epochs | lambda_ctc | Dec layers |
|---|---|---|---|---|
| `stage9a_abl_d2`   | shallower decoder       | 80  | 0.3 | **2** |
| `stage9a_abl_d4`   | deeper decoder          | 80  | 0.3 | **4** |
| `stage9a_abl_l03`  | lambda control (re-train) | 80 | 0.3 | 3 |
| `stage9a_abl_l01`  | more attention weight   | 80  | **0.1** | 3 |
| `stage9a_abl_l05`  | balanced                | 80  | **0.5** | 3 |
| `stage9a_abl_l07`  | more CTC weight         | 80  | **0.7** | 3 |
| `stage9a_abl_e120` | longer training         | **120** | 0.3 | 3 |

**Wall-clock**: ~6 h 15 m on Kaggle T4.  Fits one session with margin.

**Loop order**: cheap-first (`d2 -> d4 -> l03 -> l01 -> l05 -> l07 -> e120`) so a session timeout still leaves you with the most-informative rows.  Resume-aware via `stage9a_ablations_results.json`.

**Prerequisites**: attach a notebook-output dataset that provides `skeleton_features_t32.pt` + `subject_cv5.json`.  Checkpoints from the headline run are NOT needed for these ablations — every variant trains from scratch.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

import torch
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## Cell 2 — Locate cache + manifest, set up paths

In [ ]:
import os, glob, json, shutil
def _first(pattern):
    m = (glob.glob(f'/kaggle/working/**/{pattern}', recursive=True)
         + glob.glob(f'/kaggle/input/**/{pattern}',  recursive=True))
    return m[0] if m else None

CACHE_PATH        = _first('skeleton_features_t32.pt')
CV_MANIFEST_FOUND = _first('subject_cv5.json')
RESULTS_FOUND     = _first('stage9a_ablations_results.json')

OUT_CACHE    = CACHE_PATH        or '/kaggle/working/skeleton_features_t32.pt'
OUT_MANIFEST = CV_MANIFEST_FOUND or '/kaggle/working/subject_cv5.json'
RESULTS_PATH = '/kaggle/working/stage9a_ablations_results.json'
if RESULTS_FOUND and not os.path.exists(RESULTS_PATH):
    shutil.copy(RESULTS_FOUND, RESULTS_PATH)

CKPT_DIR = '/kaggle/working/checkpoints'
LOG_DIR  = '/kaggle/working/logs'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

print(f'cache    : {OUT_CACHE}  (exists={os.path.exists(OUT_CACHE)})')
print(f'manifest : {OUT_MANIFEST}  (exists={os.path.exists(OUT_MANIFEST)})')
print(f'results  : {RESULTS_PATH}  (exists={os.path.exists(RESULTS_PATH)})')
assert os.path.exists(OUT_CACHE),    'Attach a dataset with skeleton_features_t32.pt'
assert os.path.exists(OUT_MANIFEST), 'Attach a dataset with subject_cv5.json'

## Cell 3 — Config (locked Stage 9a hyperparams + ablation matrix)

In [ ]:
import logging, random
import numpy as np
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(LOG_DIR, 'stage9a_ablations.log'))])

from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

# Locked from Stage 9a.
T_NATIVE     = 32
UPSAMPLE     = 2
D_MODEL      = 256
N_LAYERS     = 4
N_HEADS      = 4
CONV_KERNEL  = 15
DROPOUT      = 0.2
BATCH_SIZE   = 32
LR_PEAK      = 5e-4
WEIGHT_DECAY = 5e-2
GRAD_CLIP    = 1.0
WARMUP_PCT   = 0.05
SEED         = 42
DEC_N_HEADS  = 4

# Ablation knobs (only one moves per variant).
ABLATION_FOLD = 0   # Stage 9a fold 0 was the hardest (0.5681) — most sensitive

ABLATIONS = [
    # (variant_name, num_epochs, lambda_ctc, dec_n_layers)  ordered cheap-first
    ('stage9a_abl_d2',    80,  0.3, 2),
    ('stage9a_abl_d4',    80,  0.3, 4),
    ('stage9a_abl_l03',   80,  0.3, 3),  # control (matches headline recipe)
    ('stage9a_abl_l01',   80,  0.1, 3),
    ('stage9a_abl_l05',   80,  0.5, 3),
    ('stage9a_abl_l07',   80,  0.7, 3),
    ('stage9a_abl_e120', 120,  0.3, 3),  # longest, run last
]

cfg = Config(
    data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english',
                    max_zips=None, max_frames=64, seed=SEED),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(num_epochs=80, batch_size=BATCH_SIZE, lr=LR_PEAK,
                      weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                      num_workers=2, warmup_pct=WARMUP_PCT, seed=SEED,
                      checkpoint_dir=CKPT_DIR),
).build()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f'Device         : {cfg.device}')
print(f'Ablation fold  : {ABLATION_FOLD}')
print(f'Variants       : {len(ABLATIONS)}')

## Cell 4 — Run the ablation matrix

Each variant trains from scratch on fold 0.  Cheap variants first; the 120-epoch run is last so it gets pre-empted before the cheaper rows do if a session ends abruptly.

In [ ]:
from wita_v2.training.stage9_train import train_one_fold
from wita_v2.datasets.cv_splits     import fold_indices, load_cv5_manifest
from wita_v2.datasets.skeleton_augment import LandmarkAugment

cache    = torch.load(OUT_CACHE, map_location='cpu', weights_only=False)
manifest = load_cv5_manifest(OUT_MANIFEST)

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    completed = {(r['fold'], r['variant']) for r in all_results}
    print(f'Resuming — {len(completed)} ablations already complete.')
else:
    all_results = []
    completed = set()

train_aug = LandmarkAugment()                          # Stage 1 v2 defaults
train_idx, val_idx = fold_indices(manifest, ABLATION_FOLD, cache['subjects'])
print(f'fold {ABLATION_FOLD}: train_clips={len(train_idx)}  val_clips={len(val_idx)}\n')

for variant_name, num_epochs, lambda_ctc, dec_n_layers in ABLATIONS:
    if (ABLATION_FOLD, variant_name) in completed:
        print(f'[skip] {variant_name} (already done)'); continue
    print(f'\n>>> {variant_name}: epochs={num_epochs} lambda_ctc={lambda_ctc} dec_layers={dec_n_layers}')
    result = train_one_fold(
        cache=cache, train_idx=train_idx, val_idx=val_idx, cfg=cfg,
        fold=ABLATION_FOLD, variant=variant_name,
        num_epochs=num_epochs, batch_size=BATCH_SIZE, lr_peak=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, dropout=DROPOUT,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        conv_kernel=CONV_KERNEL, upsample=UPSAMPLE, warmup_pct=WARMUP_PCT,
        dec_n_layers=dec_n_layers, dec_n_heads=DEC_N_HEADS,
        lambda_ctc=lambda_ctc,
        transform=train_aug,
        checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR,
    )
    summary = {k: v for k, v in result.items() if k != 'history'}
    # Stamp the ablation knobs into the summary for the report.
    summary['_abl_num_epochs']   = num_epochs
    summary['_abl_lambda_ctc']   = lambda_ctc
    summary['_abl_dec_n_layers'] = dec_n_layers
    all_results.append(summary)
    with open(RESULTS_PATH, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  best CER = {result["best_val_cer"]:.4f}  (best epoch {result["best_epoch"]})')
print(f'\n{len(all_results)}/{len(ABLATIONS)} ablations done.')

## Cell 5 — Ablation tables

In [ ]:
import json
with open(RESULTS_PATH) as f:
    all_results = json.load(f)
by_variant = {r['variant']: r for r in all_results}

print('=== Decoder depth ===')
print(' depth   best CER    best epoch')
for v, lbl in [('stage9a_abl_d2', '2'), ('stage9a_abl_l03', '3 (control)'), ('stage9a_abl_d4', '4')]:
    if v in by_variant:
        r = by_variant[v]
        print(f'  {lbl:<11}  {r["best_val_cer"]:.4f}      {r["best_epoch"]}')

print('\n=== lambda_ctc sweep (decoder=3 layers, 80 epochs) ===')
print(' lambda  best CER    best epoch    final ctc nll    final attn nll')
for v, lbl in [('stage9a_abl_l01', '0.1'), ('stage9a_abl_l03', '0.3'),
               ('stage9a_abl_l05', '0.5'), ('stage9a_abl_l07', '0.7')]:
    if v in by_variant:
        r = by_variant[v]
        print(f'  {lbl:<7}  {r["best_val_cer"]:.4f}      {r["best_epoch"]:<13}  '
              f'{r["final_train_ctc_nll"]:.4f}            {r["final_train_attn_nll"]:.4f}')

print('\n=== Training length ===')
for v, lbl in [('stage9a_abl_l03', '80 ep (control)'), ('stage9a_abl_e120', '120 ep')]:
    if v in by_variant:
        r = by_variant[v]
        print(f'  {lbl:<18}  best CER {r["best_val_cer"]:.4f}  (best epoch {r["best_epoch"]})')

## Cell 6 — λ_ctc curve and per-signer breakdown

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.0))

# ----- lambda curve -----
lam = []; cer = []
for v, l in [('stage9a_abl_l01', 0.1), ('stage9a_abl_l03', 0.3),
             ('stage9a_abl_l05', 0.5), ('stage9a_abl_l07', 0.7)]:
    if v in by_variant:
        lam.append(l); cer.append(by_variant[v]['best_val_cer'])
ax = axes[0]
ax.plot(lam, cer, 'o-', color='#1f77b4', markersize=8)
for x, y in zip(lam, cer):
    ax.annotate(f'{y:.4f}', (x, y), fontsize=8,
                xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('lambda_ctc')
ax.set_ylabel(f'fold {ABLATION_FOLD} best val CER')
ax.set_title('lambda_ctc sweep (decoder=3 layers, 80 epochs)')
ax.grid(True, linestyle=':', alpha=0.4)
ax.set_xticks([0.1, 0.3, 0.5, 0.7])

# ----- depth curve -----
depth = []; cer_d = []
for v, d in [('stage9a_abl_d2', 2), ('stage9a_abl_l03', 3), ('stage9a_abl_d4', 4)]:
    if v in by_variant:
        depth.append(d); cer_d.append(by_variant[v]['best_val_cer'])
ax = axes[1]
ax.plot(depth, cer_d, 's-', color='#d62728', markersize=8)
for x, y in zip(depth, cer_d):
    ax.annotate(f'{y:.4f}', (x, y), fontsize=8,
                xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('decoder layers')
ax.set_ylabel(f'fold {ABLATION_FOLD} best val CER')
ax.set_title('decoder depth (lambda_ctc=0.3, 80 epochs)')
ax.set_xticks([2, 3, 4])
ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'stage9a_ablations_curves.png'), dpi=140)
plt.show()

## Cell 7 — Decisions to make from these numbers

1. **If `stage9a_abl_e120` < `stage9a_abl_l03` by ≥ 0.02**: re-run the full 5-fold headline at 120 epochs.  Costs another ~6 h but raises the headline.
2. **If `stage9a_abl_l05` or `stage9a_abl_l07` < `stage9a_abl_l03` by ≥ 0.02**: re-run the 5-fold headline at that lambda.  Reviewer-grade improvement.
3. **If the depth curve has a clear minimum at depth=4**: re-run headline with 4-layer decoder.
4. **If everything is within ±0.015 of the control**: the chosen Stage 9a recipe was at or near the local optimum, no headline re-run needed; ablations stay in the appendix.

Commit the kernel either way so the appendix figures and per-fold detail are preserved.